In [ ]:
# 1. NSMC 데이터 다운로드 및 데이터 구성
# NSMC train 데이터를 다운로드해주세요.
import tensorflow as tf
import numpy as np
import pandas as pd
!pip install evaluate

path_to_train_file = tf.keras.utils.get_file(
    'train.txt',
    'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt'
)

# 리뷰 문장(text)과 감정 라벨(label)을 각각 분리해주세요.
train_text = open(path_to_train_file, 'rb').read().decode(encoding='utf-8')

train_X= np.array([
    row.split('\t')[1]
    for row in train_text.split('\n')[1:]
    if row.count('\t') > 0
])

train_Y = np.array([
    int(row.split('\t')[2])
    for row in train_text.split('\n')[1:]
    if row.count('\t') > 0
])

# text와 label 형태로 데이터를 구성해주세요. 모델 구성에 일부 데이터만 사용해도 괜찮습니다.
train_df = pd.DataFrame({
    'text': train_X,
    'label': train_Y
})

train_df = train_df.sample(1000, random_state=42).reset_index(drop=True)

print(train_df.head())
print(train_df.shape)



In [ ]:

# 2. HuggingFace 사전학습 모델 불러오기
# transformers 라이브러리를 사용해주세요.한국어 BERT 계열 모델을 사용해주세요.
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "klue/bert-base"

# Tokenizer와 Sequence Classification 모델을 각각 불러와주세요.감정 분석은 긍정/부정 분류이므로 label=2로 설정해주세요.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)




In [ ]:
# 3. Dataset 구성하기
# HuggingFace datasets 라이브러리의 Dataset 객체를 사용해주세요.

from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df)

# text와 label이 포함된 Dataset을 구성해주세요.

train_dataset = Dataset.from_pandas(train_df)

# train / validation 데이터를 분리해주세요.
split_dataset = train_dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

In [ ]:
# 4. Tokenizing 함수 구현하기
# tokenizer를 사용하여 문장을 tokenizing 해주세요.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#`truncation` 옵션을 설정하여 너무 긴 문장이 잘리도록 해주세요.
# `padding` 옵션을 설정하여 문장 길이를 일정하게 맞춰주세요.
# `max_length` 값을 직접 지정해주세요.
MAX_LEN = 32

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_val_dataset = val_dataset.map(tokenize_function, batched=True)

In [ ]:
# 5. Fine-Tuning 학습 구성하기
# Fine-Tuning 학습을 위한 하이퍼파라미터를 설정해주세요.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=20
)

#학습(train) 데이터와 검증(validation) 데이터를 활용하여 모델을 학습해주세요.
from transformers import Trainer
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1);
    return accuracy.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()


In [ ]:
# 6. 모델 성능 평가하기
# validation dataset 기준으로 성능을 평가해주세요.

eval_result = trainer.evaluate(eval_dataset=tokenized_val_dataset)

print(eval_result)

# Accuracy 또는 이에 준하는 metric을 출력해주세요.
print("Validation Accuracy:", eval_result["eval_accuracy"])


In [ ]:
# 7. 감정 분석 테스트하기
# Fine-Tuning이 완료된 모델을 사용하여 실제 문장을 예측해주세요.
# 최소 3개의 문장을 직접 테스트해주세요.
# 긍정/부정 결과를 출력해주세요.

import torch

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=64
    )

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        predicted_class = torch.argmax(logits, dim=1).item()

    if predicted_class == 1:
        return "긍정"
    else:
        return "부정"

print(predict_sentiment("이 영화 너무 감동적이고 재밌었어요"))
print(predict_sentiment("정말 지루하고 돈 아까운 영화였습니다"))
print(predict_sentiment("결말이 너무 아름다운 영화에요 시간 가는 줄도 모르고 봤습니다"))

In [ ]:
# 다시 Colab으로 돌아와서 실행
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import login

login()

MY_MODEL_NAME = "gyk1229/nsmc-sentiment"

model.push_to_hub(MY_MODEL_NAME)
tokenizer.push_to_hub(MY_MODEL_NAME)

print("업로드 완료!")
print(f"모델 주소: https://huggingface.co/{MY_MODEL_NAME}")

In [ ]:
from huggingface_hub import HfApi

MY_MODEL_NAME = "gyk1229/nsmc-sentiment"
MY_SPACE_NAME = "gyk1229/nsmc-sentiment-demo"

# HuggingFace Hub에 업로드한 감정 분석 모델을 불러와 Gradio 웹앱으로 실행하는 코드입니다.
# Gradio UI 코드 부분은 자유롭게 구성하셔도 됩니다.
app_code = f'''
import gradio as gr
from transformers import pipeline

classifier = pipeline("text-classification", model="{MY_MODEL_NAME}")

def format_result(result):
    label = result["label"]
    score = result["score"]

    if label == "LABEL_1":
        emoji, label_kr = "😊", "긍정"
    else:
        emoji, label_kr = "😞", "부정"

    return f"{{emoji}} {{label_kr}} (확신도: {{score:.1%}})"

def predict(text):
    if not text.strip():
        return "문장을 입력해주세요."
    result = classifier(text)[0]
    return format_result(result)

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(label="영화 리뷰", placeholder="리뷰를 입력하세요...", lines=3),
    outputs=gr.Textbox(label="감정 분석 결과"),
    title="AI 영화 리뷰 감정 분석기",
    description="NSMC 데이터로 파인튜닝된 한국어 감정 분석 모델입니다.",
    examples=[
        ["이 영화 진짜 재미있어요!"],
        ["완전 지루하고 별로였음"],
        ["배우 연기는 좋았지만 스토리가 아쉬웠다"]
    ]
)
demo.launch()
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

with open('requirements.txt', 'w') as f:
    f.write("transformers\ngradio\ntorch\n")

# Spaces에 업로드
api = HfApi()

# 먼저 Hugging Face Space 저장소를 생성합니다.
# private=True 로 설정하면 비공개 저장소로 생성됩니다.
api.create_repo(repo_id=MY_SPACE_NAME, repo_type="space", private=False, exist_ok=True, space_sdk="gradio")

api.upload_file(
    path_or_fileobj="app.py",
    path_in_repo="app.py",
    repo_id=MY_SPACE_NAME,
    repo_type="space"
)
api.upload_file(
    path_or_fileobj="requirements.txt",
    path_in_repo="requirements.txt",
    repo_id=MY_SPACE_NAME,
    repo_type="space"
)

print("완료!")
print(f"https://huggingface.co/spaces/{MY_SPACE_NAME}")